In [76]:
import pandas as pd
import os
import requests
import numpy as np
import config
import subprocess
from io import StringIO
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from pathlib import Path
from get_elo import update_elo_history

In [77]:
GIT_REPO = os.path.expanduser("~/MyProjects/FPL/FPL-Elo-Insights")
subprocess.run(["git", "-C", GIT_REPO, "pull"], check=True)

def get_gw_state():

    
    return requests.get("https://draft.premierleague.com/api/game").json()
state = get_gw_state()
GW = state["next_event"]
LAST_GW = GW - 1
TOTAL_MINUTES = GW * 90

last_year_stats = pd.read_excel("seasons/2025-26/End2526_Full.xlsx")

FPL_DATA_FOLDER = os.path.expanduser('~/MyProjects/FPL/FPL-Elo-Insights/data/2026-2027')

fixtures_dir = os.path.join(FPL_DATA_FOLDER, "By Gameweek")
players = pd.read_csv(f"{FPL_DATA_FOLDER}/players.csv",encoding="utf-8")
teams = pd.read_csv(f"{FPL_DATA_FOLDER}/teams.csv",encoding="utf-8")

playerstats = pd.read_csv(f"{FPL_DATA_FOLDER}/By Gameweek/GW{LAST_GW}/playerstats.csv",encoding="utf-8")

hist = update_elo_history(GW)
elo = hist[hist["date"] == hist["date"].max()].set_index("code")["elo"]

teams = teams.drop(columns=["elo"])
teams = teams.merge(elo, left_on="code", right_index=True, how="left")
print(teams.head())


From https://github.com/olbauday/FPL-Core-Insights
   c023e79b6..978d93999  main       -> origin/main


Updating c023e79b6..978d93999
Fast-forward
 data/2026-2027/By Gameweek/GW10/players.csv        |  372 +++---
 data/2026-2027/By Gameweek/GW11/players.csv        |  372 +++---
 data/2026-2027/By Gameweek/GW12/players.csv        |  372 +++---
 data/2026-2027/By Gameweek/GW13/players.csv        |  372 +++---
 data/2026-2027/By Gameweek/GW14/players.csv        |  372 +++---
 data/2026-2027/By Gameweek/GW15/players.csv        |  372 +++---
 data/2026-2027/By Gameweek/GW16/players.csv        |  372 +++---
 data/2026-2027/By Gameweek/GW17/players.csv        |  372 +++---
 data/2026-2027/By Gameweek/GW18/players.csv        |  372 +++---
 data/2026-2027/By Gameweek/GW19/players.csv        |  372 +++---
 data/2026-2027/By Gameweek/GW2/fixtures.csv        |    2 +-
 data/2026-2027/By Gameweek/GW2/matches.csv         |    2 +-
 data/2026-2027/By Gameweek/GW20/players.csv        |  372 +++---
 data/2026-2027/By Gameweek/GW21/players.csv        |  372 +++---
 data/2026-2027/By Gameweek/GW22/players.

In [78]:
# 1. playerstats + players (get names, position, team_code)
merged = playerstats.merge(
    players,
    left_on="id",
    right_on="player_id",
    how="left",
    suffixes=("", "_p")      # keep playerstats names clean, suffix player dups
)

# 2. + teams (get team short_name)
merged = merged.merge(
    teams[["code", "short_name"]],
    left_on="team_code",
    right_on="code",
    how="left"
).rename(columns={"short_name": "team_name"}).drop(columns=["code"])


base_cols = [
    # --- identity ---
    "web_name",
    "position",
    "team_name",

    # --- core output / value ---
    "total_points",
    "points_per_game",
    "minutes",
    "starts",
    "bonus",
    "bps",
    "form",

    # --- expected (xG) ---
    "expected_goals_per_90",
    "expected_assists_per_90",
    "expected_goal_involvements_per_90",
    "expected_goals_conceded_per_90",

    # --- defcon ---
    "defensive_contribution",
    "defensive_contribution_per_90",
    "tackles",
    "clearances_blocks_interceptions",
    "recoveries",

    # --- set pieces ---
    "penalties_order",
    "corners_and_indirect_freekicks_order",
    "direct_freekicks_order",

    # --- gk-specific ---
    "saves_per_90",
    "clean_sheets_per_90",
    "goals_conceded_per_90",

    # meta
    "player_id",
    "team_code",

    # --- availability ---
    "status",
    "chance_of_playing_next_round"
]

base = merged[base_cols].copy()

base = base.sort_values(by='total_points', ascending=False)

In [79]:
all_fixtures = []

for folder in sorted(os.listdir(fixtures_dir)):
    if folder.startswith("GW"):
        gw_path = os.path.join(fixtures_dir, folder, "fixtures.csv")
        if os.path.exists(gw_path):
            df = pd.read_csv(gw_path, encoding="utf-8")
            all_fixtures.append(df)

fixtures_master = pd.concat(all_fixtures, ignore_index=True)
fixtures_master["tournament"] = fixtures_master["tournament"].fillna("")
fixtures_master = fixtures_master[
    fixtures_master["tournament"].str.lower().str.contains("prem")
]

In [80]:
# #### FIXTURE GAMBLER MACHINE LEARNING MODEL

fixtures_master = fixtures_master.copy()

# ELO going INTO each GW:  (GW1-6 backfilled, estimated) + scraped history (GW7+)

elo_all = pd.concat([pd.read_csv("data/elo_backfill.csv"),
                     pd.read_csv("data/elo_history.csv")])
elo_all = elo_all.drop_duplicates(["gw", "code"], keep="last")

elo_long = (elo_all.pivot(index="gw", columns="code", values="elo")
            .reindex(range(1, 39)).ffill() 
            .stack())

for side in ["home", "away"]:
    idx = pd.MultiIndex.from_arrays([fixtures_master["gameweek"], fixtures_master[f"{side}_team"]])
    fixtures_master[f"{side}_team_elo"] = elo_long.reindex(idx).values

fixtures_master["elo_diff"] = fixtures_master["home_team_elo"] - fixtures_master["away_team_elo"]
fixtures_master["elo_context"] = fixtures_master["elo_diff"] / 250

played  = fixtures_master[fixtures_master["gameweek"] <= GW].copy()

# ✅ Defensive stats
played["home_non_penalty_xg_allowed"] = played["away_non_penalty_xg"]
played["away_non_penalty_xg_allowed"] = played["home_non_penalty_xg"]

played["home_xg_set_play_allowed"] = played["away_xg_set_play"]
played["away_xg_set_play_allowed"] = played["home_xg_set_play"]

played["home_goals_conceded"] = played["away_score"]
played["away_goals_conceded"] = played["home_score"]

played["home_points"] = played["home_score"].gt(played["away_score"])*3 + \
                                 played["home_score"].eq(played["away_score"])*1
played["away_points"] = played["away_score"].gt(played["home_score"])*3 + \
                                 played["away_score"].eq(played["home_score"])*1

played["home_xg_on_target_xgot_allowed"] = played["away_xg_on_target_xgot"]
played["away_xg_on_target_xgot_allowed"] = played["home_xg_on_target_xgot"]

played["home_keeper_saves_allowed"] = played["away_keeper_saves"]
played["away_keeper_saves_allowed"] = played["home_keeper_saves"]

fixtures = played.copy()

### CREATING ROLLING METRICS

# ---- HOME ----
home = fixtures[[
    "gameweek",
    "home_team",
    "home_team_elo",
    "home_score",
    "home_goals_conceded",
    "home_points",
    "home_non_penalty_xg",
    "home_xg_set_play",
    "home_non_penalty_xg_allowed",
    "home_xg_set_play_allowed",
    "home_xg_on_target_xgot",
    "home_xg_on_target_xgot_allowed",
    "home_keeper_saves"
]].copy()

home.columns = [
    "gameweek",
    "team",
    "elo",
    "goals_scored",
    "goals_conceded",
    "points",
    "non_penalty_xg",
    "xg_set_play",
    "non_penalty_xg_allowed",
    "xg_set_play_allowed",
    "xg_on_target_xgot",
    "xg_on_target_xgot_allowed",
    "keeper_saves",
]
home["is_home"] = 1

# ---- AWAY ----
away = fixtures[[
    "gameweek",
    "away_team",
    "away_team_elo",
    "away_score",
    "away_goals_conceded",
    "away_points",
    "away_non_penalty_xg",
    "away_xg_set_play",
    "away_non_penalty_xg_allowed",
    "away_xg_set_play_allowed",
    "away_xg_on_target_xgot",
    "away_xg_on_target_xgot_allowed",
    "away_keeper_saves"
]].copy()

away.columns = [
    "gameweek",
    "team",
    "elo",
    "goals_scored",
    "goals_conceded",
    "points",
    "non_penalty_xg",
    "xg_set_play",
    "non_penalty_xg_allowed",
    "xg_set_play_allowed",
    "xg_on_target_xgot",
    "xg_on_target_xgot_allowed",
    "keeper_saves",
]
away["is_home"] = 0

team_fixtures = pd.concat([home, away], ignore_index=True)
team_fixtures = team_fixtures.sort_values(["team","gameweek"])

# Finishing: are they turning xG into good shots?
team_fixtures["finishing_xgot_over"] = (
    team_fixtures["xg_on_target_xgot"] - team_fixtures["non_penalty_xg"]
)

# Defense / GK: are they conceding less/more than the xGOT they face?
team_fixtures["shot_stopping_diff"] = (
    team_fixtures["xg_on_target_xgot_allowed"] - team_fixtures["goals_conceded"]
)

bad = team_fixtures[team_fixtures.xg_set_play > team_fixtures.non_penalty_xg]
print("badtimes")
print(len(bad)); print(bad[["team","gameweek","non_penalty_xg","xg_set_play"]])

# a simple keeper quality proxy
team_fixtures["keeper_intensity"] = (
    team_fixtures["keeper_saves"] /
    (team_fixtures["xg_on_target_xgot_allowed"].clip(lower=0.1))
)

team_fixtures["open_play_xg"]         = team_fixtures["non_penalty_xg"] - team_fixtures["xg_set_play"]
team_fixtures["open_play_xg_allowed"] = team_fixtures["non_penalty_xg_allowed"] - team_fixtures["xg_set_play_allowed"]

def roll(col, n, by=("team",)):
    return (team_fixtures.groupby(list(by))[col]
            .transform(lambda s: s.shift(1).rolling(n, min_periods=1).mean()))

for n in (3, 5):
    team_fixtures[f"roll{n}_xg_set_play"]         = roll("xg_set_play", n)
    team_fixtures[f"roll{n}_xg_set_play_allowed"] = roll("xg_set_play_allowed", n)
    team_fixtures[f"roll{n}_open_play_xg"]         = roll("open_play_xg", n)
    team_fixtures[f"roll{n}_open_play_xg_allowed"] = roll("open_play_xg_allowed", n)
    # home/away-split actuals
    team_fixtures[f"roll{n}_finishing_over"]       = roll("finishing_xgot_over", n, ("team", "is_home"))
    team_fixtures[f"roll{n}_shot_stopping"]        = roll("shot_stopping_diff", n, ("team", "is_home"))
    team_fixtures[f"roll{n}_keeper_intensity"]     = roll("keeper_intensity", n, ("team", "is_home"))

# a team's first home (or away) game has no prior same-venue game -> NaN; treat as neutral
for c in ["finishing_over", "shot_stopping", "keeper_intensity"]:
    for n in (3, 5):
        team_fixtures[f"roll{n}_{c}"] = team_fixtures[f"roll{n}_{c}"].fillna(0)

### WEIGHTED FORM

def blend(c): return 0.7 * team_fixtures[f"roll3_{c}"] + 0.3 * team_fixtures[f"roll5_{c}"]

team_fixtures["weighted_set_play_strength"]  = blend("xg_set_play")
team_fixtures["weighted_open_play_strength"] = blend("open_play_xg")
team_fixtures["weighted_set_play_defence"]   = blend("xg_set_play_allowed")
team_fixtures["weighted_open_play_defence"]  = blend("open_play_xg_allowed")

team_fixtures["weighted_attack"] = (
    0.50 * blend("open_play_xg") + 0.25 * blend("xg_set_play") + 0.25 * blend("finishing_over"))

team_fixtures["weighted_defence"] = (
    -0.50 * blend("open_play_xg_allowed") - 0.25 * blend("xg_set_play_allowed")
    + 0.20 * blend("shot_stopping") + 0.05 * blend("keeper_intensity"))

latest = team_fixtures.sort_values("gameweek").groupby("team").tail(1)
latest = latest.merge(teams[["code","short_name"]], left_on="team", right_on="code", how="left")


print("set-play subset:", (team_fixtures.xg_set_play <= team_fixtures.non_penalty_xg).all())
print(latest[["short_name","weighted_attack"]].sort_values("weighted_attack", ascending=False).round(2).to_string(index=False))
print(latest[["short_name","weighted_defence"]].sort_values("weighted_defence", ascending=False).round(2).to_string(index=False))
print(latest[["short_name","elo"]].sort_values("elo", ascending=False).to_string(index=False))
print(fixtures_master[["home_team_elo","away_team_elo"]].isna().sum())

bad = team_fixtures[team_fixtures.xg_set_play > team_fixtures.non_penalty_xg]
print("badtimes")
print(len(bad)); print(bad[["team","gameweek","non_penalty_xg","xg_set_play"]])

badtimes
0
Empty DataFrame
Columns: [team, gameweek, non_penalty_xg, xg_set_play]
Index: []
set-play subset: False
short_name  weighted_attack
       ARS             0.81
       IPS             0.80
       NFO             0.74
       MCI             0.74
       EVE             0.71
       NEW             0.67
       LIV             0.67
       BRE             0.63
       LEE             0.63
       BHA             0.59
       HUL             0.58
       SUN             0.57
       TOT             0.56
       FUL             0.55
       AVL             0.55
       CRY             0.52
       BOU             0.51
       MUN             0.44
       CHE             0.39
       COV             0.36
short_name  weighted_defence
       LIV             -0.19
       AVL             -0.22
       ARS             -0.29
       EVE             -0.30
       HUL             -0.32
       BHA             -0.36
       BRE             -0.37
       LEE             -0.40
       COV             -0.40
       

In [81]:
from sklearn.linear_model import PoissonRegressor

def side(df, a, b, hm):
    return pd.DataFrame({"gw": df["gameweek"], "team": df[f"{a}_team"], "opp": df[f"{b}_team"],
                         "home": hm, "y": df[f"{a}_non_penalty_xg"],
                         "elo": (df[f"{a}_team_elo"] - df[f"{b}_team_elo"]) / 250})

fm = fixtures_master[fixtures_master["gameweek"] <= GW]
long = pd.concat([side(fm, "home", "away", 1), side(fm, "away", "home", 0)], ignore_index=True)
train = long[long["y"].notna()]

def design(df, cols=None):
    X = pd.get_dummies(df[["team", "opp"]].astype(int).astype(str), prefix=["att", "def"]).astype(float)
    X["home"], X["elo"] = df["home"].values, df["elo"].values
    return X if cols is None else X.reindex(columns=cols, fill_value=0)

HALF, ALPHA = 6, 1.0          # to tune
Xtr = design(train)
model = PoissonRegressor(alpha=ALPHA, max_iter=1000).fit(
    Xtr, train["y"], sample_weight=0.5 ** ((GW - train["gw"]) / HALF))

from sklearn.metrics import mean_poisson_deviance

long = long.reset_index(drop=True)
X_all = design(long)                    # dummies built once for all teams
played_mask = long["y"].notna()

def walk(HALF, ALPHA, first=3):
    out = []
    for k in range(first, GW):          # GW itself is unplayed, so test on 3..GW-1
        tr = played_mask & (long["gw"] < k)
        te = played_mask & (long["gw"] == k)
        w = 0.5 ** ((k - long.loc[tr, "gw"]) / HALF)
        m = PoissonRegressor(alpha=ALPHA, max_iter=1000).fit(X_all[tr], long.loc[tr, "y"], sample_weight=w)
        p = np.clip(m.predict(X_all[te]), 0.05, None)
        y = long.loc[te, "y"]
        base = np.full(len(y), np.average(long.loc[tr, "y"], weights=w))
        out.append((mean_poisson_deviance(y, p), mean_poisson_deviance(y, base)))
    return np.mean(out, axis=0)

rows = [(h, a, *walk(h, a)) for h in (3, 6, 12, 1e9) for a in (0.3, 1, 3, 10)]
res = pd.DataFrame(rows, columns=["half", "alpha", "model", "baseline"])
res["gain"] = res["baseline"] - res["model"]       # lower deviance is better
print(res.sort_values("gain", ascending=False).round(3).to_string(index=False))

        half  alpha  model  baseline   gain
1000000000.0    3.0  0.319     0.322  0.002
        12.0    3.0  0.320     0.322  0.002
         6.0    3.0  0.320     0.322  0.002
         3.0    3.0  0.321     0.323  0.002
1000000000.0   10.0  0.320     0.322  0.001
        12.0   10.0  0.321     0.322  0.001
         6.0   10.0  0.321     0.322  0.001
         3.0   10.0  0.322     0.323  0.001
1000000000.0    1.0  0.322     0.322 -0.001
        12.0    1.0  0.323     0.322 -0.001
         6.0    1.0  0.323     0.322 -0.001
         3.0    1.0  0.324     0.323 -0.001
         3.0    0.3  0.346     0.323 -0.023
         6.0    0.3  0.346     0.322 -0.024
        12.0    0.3  0.346     0.322 -0.024
1000000000.0    0.3  0.346     0.322 -0.025


In [82]:
home_features = (
    team_fixtures[team_fixtures["is_home"] == 1]
    .rename(columns=lambda c: "home_" + c if c not in ["gameweek", "team", "is_home"] else c)
)

away_features = (
    team_fixtures[team_fixtures["is_home"] == 0]
    .rename(columns=lambda c: "away_" + c if c not in ["gameweek", "team", "is_home"] else c)
)

played = played.merge(
    home_features[["gameweek","team"] + [c for c in home_features.columns if c.startswith("home_")]],
    left_on=["gameweek","home_team"], right_on=["gameweek","team"], how="left"
).drop(columns=["team"])

played = played.merge(
    away_features[["gameweek","team"] + [c for c in away_features.columns if c.startswith("away_")]],
    left_on=["gameweek","away_team"], right_on=["gameweek","team"], how="left"
).drop(columns=["team"])

print([c for c in played.columns if "weighted" in c])
print([c for c in team_fixtures.columns if "weighted" in c])



['home_weighted_set_play_strength', 'home_weighted_open_play_strength', 'home_weighted_set_play_defence', 'home_weighted_open_play_defence', 'home_weighted_attack', 'home_weighted_defence', 'away_weighted_set_play_strength', 'away_weighted_open_play_strength', 'away_weighted_set_play_defence', 'away_weighted_open_play_defence', 'away_weighted_attack', 'away_weighted_defence']
['weighted_set_play_strength', 'weighted_open_play_strength', 'weighted_set_play_defence', 'weighted_open_play_defence', 'weighted_attack', 'weighted_defence']


In [83]:

played["set_play_diff"] = (
    (played["home_weighted_set_play_strength"] -
     played["away_weighted_set_play_defence"]) -
    (played["away_weighted_set_play_strength"] -
     played["home_weighted_set_play_defence"])
)

played["open_play_diff"] = (
    (played["home_weighted_open_play_strength"] -
     played["away_weighted_open_play_defence"]) -
    (played["away_weighted_open_play_strength"] -
     played["home_weighted_open_play_defence"])
)

played["attack_diff"] = (
    (played["home_weighted_attack"] - played["away_weighted_defence"]) -
    (played["away_weighted_attack"] - played["home_weighted_defence"])
)

played = played.sort_values("gameweek")

train = played[played["gameweek"] < GW]
test  = played[played["gameweek"] == GW]

print(test.shape)                              

FEATURES = ["attack_diff", "set_play_diff", "open_play_diff", "elo_context"] 


train = train.dropna(subset=FEATURES + ["home_score", "away_score"])

test  = test.dropna(subset=FEATURES)

print("wake up")
print("elo_context" in train.columns, train["elo_context"].isna().sum(), test["elo_context"].isna().sum())

print(dict(zip(FEATURES, model.coef_.round(3))))
X = train[FEATURES]
X_test = test[FEATURES]

print(X_test.head(20))

y_home = train["home_score"]
y_away = train["away_score"]

### FIXTURE PREDICTOR
model_home = LinearRegression().fit(X, y_home)
model_away = LinearRegression().fit(X, y_away)

train_home_pred = model_home.predict(X)
train_away_pred = model_away.predict(X)

mu_home  = train_home_pred.mean()
std_home = train_home_pred.std()

mu_away  = train_away_pred.mean()
std_away = train_away_pred.std()

test["pred_home_goals"] = model_home.predict(X_test)
test["pred_away_goals"] = model_away.predict(X_test)

test["pred_goal_diff"] = test["pred_home_goals"] - test["pred_away_goals"]

## SCORE NORMALIZATION
test["home_attack_fixture"] = test["pred_home_goals"] - mu_home
test["away_attack_fixture"] = test["pred_away_goals"] - mu_away

test["home_defence_fixture"] = mu_away - test["pred_away_goals"]
test["away_defence_fixture"] = mu_home - test["pred_home_goals"]

test = test.merge(
    teams[['code', 'short_name']],
    left_on='home_team',
    right_on='code',
    how='left'
).rename(columns={
    'short_name': 'home_team_name'
}).drop(columns=['code'])

test = test.merge(
    teams[['code', 'short_name']],
    left_on='away_team',
    right_on='code',
    how='left'
).rename(columns={
    'short_name': 'away_team_name'
}).drop(columns=['code'])

test["pred_total_goals"] = test["pred_home_goals"] + test["pred_away_goals"]
test["pred_total_goals"] = test["pred_total_goals"].round(1)

test["kickoff_time"] = pd.to_datetime(test["kickoff_time"])
test = test.sort_values("kickoff_time")

cols_order = ["home_team_name", "pred_home_goals", "pred_away_goals", "away_team_name",
                  "pred_total_goals", "pred_goal_diff", "attack_diff",
    "home_attack_fixture", "away_defence_fixture", "away_attack_fixture", "home_defence_fixture", 
    "home_weighted_attack", "away_weighted_defence", "away_weighted_attack", "home_weighted_defence",
   "set_play_diff", "open_play_diff", 
  "home_weighted_set_play_strength",     "home_weighted_open_play_strength", 
    "home_weighted_set_play_defence",     "home_weighted_open_play_defence",
     "away_weighted_set_play_strength",     "away_weighted_open_play_strength",
    "away_weighted_set_play_defence",   "away_weighted_open_play_defence"]


predictions = test[cols_order]

name_cols = ["home_team_name", "away_team_name"]
num_cols = [c for c in predictions.columns if c not in name_cols and pd.api.types.is_numeric_dtype(predictions[c])]

predictions = predictions.copy()
predictions[num_cols] = predictions[num_cols].round(1)

predictions.to_csv("predicted_fixtures_GW.csv", index=False)

def walk_cols(cols, ALPHA=0.01, first=3):
    out = []
    for k in range(first, GW):
        tr = played_mask & (long["gw"] < k); te = played_mask & (long["gw"] == k)
        m = PoissonRegressor(alpha=ALPHA, max_iter=1000).fit(X_all.loc[tr, cols], long.loc[tr, "y"])
        p = np.clip(m.predict(X_all.loc[te, cols]), 0.05, None); y = long.loc[te, "y"]
        base = np.full(len(y), long.loc[tr, "y"].mean())
        out.append((mean_poisson_deviance(y, p), mean_poisson_deviance(y, base)))
    return np.mean(out, axis=0)

print(walk_cols(["home", "elo"]), walk_cols(["home"]))

(10, 205)
wake up
True 0 0
{'attack_diff': np.float64(0.002), 'set_play_diff': np.float64(0.003), 'open_play_diff': np.float64(-0.006), 'elo_context': np.float64(0.014)}
    attack_diff  set_play_diff  open_play_diff  elo_context
60    -0.294002      -0.062867        0.450133        0.640
61     0.529089       0.107467       -0.631733        0.264
62     0.354962       0.298200       -0.045267       -0.244
63     0.297299       0.808533       -0.374733        1.332
64    -0.121040      -0.055133       -0.376000       -0.112
65     0.331030      -0.538000       -0.415200        0.476
66    -0.223116       0.225133        0.541200       -0.976
67     0.129071       0.020800       -0.315200        0.372
68     0.568788      -0.460933        0.475333       -0.264
69    -0.210479       0.481733        0.282267       -0.264
[0.3501779  0.32159434] [0.3439254  0.32159434]


In [84]:
# ---- FIXTURE SCORE from predictor (single GW) ----
home_fx = test[["home_team","home_attack_fixture","home_defence_fixture","pred_away_goals"]].rename(
    columns={"home_team":"team_code","home_attack_fixture":"fixture_attack",
             "home_defence_fixture":"fixture_defence","pred_away_goals":"opp_pred_goals"})
away_fx = test[["away_team","away_attack_fixture","away_defence_fixture","pred_home_goals"]].rename(
    columns={"away_team":"team_code","away_attack_fixture":"fixture_attack",
             "away_defence_fixture":"fixture_defence","pred_home_goals":"opp_pred_goals"})
team_fx = pd.concat([home_fx, away_fx], ignore_index=True)

# Poisson clean-sheet probability — convex boost as opponent's predicted goals drop
team_fx["cs_prob"] = np.exp(-team_fx["opp_pred_goals"].clip(lower=0))

print(team_fx["cs_prob"].describe())
print(team_fx["cs_prob"].head(10))

# normalize
team_fx["fixture_attack_score"]  = team_fx["fixture_attack"] / (team_fx["fixture_attack"].abs().max() + 1e-9)
team_fx["fixture_defence_score"] = team_fx["cs_prob"] / (team_fx["cs_prob"].max() + 1e-9)



count    20.000000
mean      0.266020
std       0.114112
min       0.058855
25%       0.189820
50%       0.247315
75%       0.321805
max       0.552729
Name: cs_prob, dtype: float64
0    0.184286
1    0.191665
2    0.552729
3    0.331018
4    0.239006
5    0.351032
6    0.359089
7    0.181451
8    0.258277
9    0.270498
Name: cs_prob, dtype: float64


In [85]:
# 1. latest team strength (most recent rolling value per team)
teams_latest = (
    team_fixtures.sort_values("gameweek")
    .groupby("team").tail(1)[["team", "weighted_attack", "weighted_defence"]]
)

# 2. rescale to friendly 0-100 (linear — preserves order + distances)
def to_friendly(s, lo=0, hi=100):
    rng = s.max() - s.min()
    return pd.Series(50, index=s.index) if rng == 0 else lo + (s - s.min()) / rng * (hi - lo)

teams_latest["team_attack_score"]  = to_friendly(teams_latest["weighted_attack"])
teams_latest["team_defence_score"] = to_friendly(teams_latest["weighted_defence"])

# 3. merge onto player board (team_code ↔ team)
merged = base.merge(
    teams_latest[["team", "team_attack_score", "team_defence_score"]],
    left_on="team_code", right_on="team", how="left"
).drop(columns=["team"])

merged["team_score"] = np.select(
    [
        merged["position"] == "Forward",
        merged["position"] == "Midfielder",
        merged["position"] == "Defender",
        merged["position"] == "Goalkeeper",
    ],
    [
        merged["team_attack_score"],                                   # FWD: pure attack
        0.65 * merged["team_attack_score"] + 0.35 * merged["team_defence_score"],  # MID: mostly attack
        merged["team_defence_score"],                                  # DEF: pure defence
        merged["team_defence_score"],                                  # GK: pure defence
    ],
    default=0.0
)

merged = merged.merge(
    team_fx[["team_code","fixture_attack_score","fixture_defence_score"]],
    on="team_code", how="left"
)

merged["fixture_score"] = np.select(
    [merged["position"]=="Forward", merged["position"]=="Midfielder",
     merged["position"]=="Defender", merged["position"]=="Goalkeeper"],
    [merged["fixture_attack_score"],
     0.65*merged["fixture_attack_score"] + 0.35*merged["fixture_defence_score"],
     merged["fixture_defence_score"],
     merged["fixture_defence_score"]],
    default=0.0
)

merged["fixture_score_100"] = to_friendly(merged["fixture_score"].fillna(0))

print(merged.columns.tolist())
print(merged[["web_name", "team_name", "team_defence_score","team_attack_score", "team_score"]].head(20))

['web_name', 'position', 'team_name', 'total_points', 'points_per_game', 'minutes', 'starts', 'bonus', 'bps', 'form', 'expected_goals_per_90', 'expected_assists_per_90', 'expected_goal_involvements_per_90', 'expected_goals_conceded_per_90', 'defensive_contribution', 'defensive_contribution_per_90', 'tackles', 'clearances_blocks_interceptions', 'recoveries', 'penalties_order', 'corners_and_indirect_freekicks_order', 'direct_freekicks_order', 'saves_per_90', 'clean_sheets_per_90', 'goals_conceded_per_90', 'player_id', 'team_code', 'status', 'chance_of_playing_next_round', 'team_attack_score', 'team_defence_score', 'team_score', 'fixture_attack_score', 'fixture_defence_score', 'fixture_score', 'fixture_score_100']
         web_name team_name  team_defence_score  team_attack_score  team_score
0       De Cuyper       BHA           76.955987          52.155480   76.955987
1            Groß       BHA           76.955987          52.155480   60.835657
2       Tarkowski       EVE           85.4

In [86]:
###LAST SEASON'S STATS

# Columns to bring from last year
cols_needed = [
    "web_name",
    "points_per_game",
    "x_fpl_p90",
    "total_points",
    "bonus_p90",
    "minutes",
    "defcon_p90"
]


# Prepare last year slice with prefix
last_year_slim = last_year_stats[cols_needed].copy()
last_year_slim = last_year_slim.rename(columns={col: f"last_{col}" for col in cols_needed if col != "web_name"})
last_year_slim = last_year_slim[last_year_slim["last_total_points"] > 0]

last_year_slim = (
    last_year_slim
    .sort_values("last_minutes", ascending=False)
    .drop_duplicates("web_name", keep="first")
)

# Merge (current is LEFT, last year secondary)
merged = merged.merge(
    last_year_slim,
    left_on="web_name",
    right_on="web_name",
    how="left",
    indicator=True
)

In [87]:
def scale_global(df, col):
    s = df[col]
    median = s.median()
    iqr = (s.quantile(0.75) - s.quantile(0.25)) + 1e-6
    return (s - median) / iqr

merged["minutes_factor"] = np.sqrt((merged["minutes"] / (LAST_GW * 90)).clip(upper=1))

print(merged["minutes_factor"].describe())

def expected_fpl_p90(row):
    pos = row["position"]
    xg = row.get("expected_goals_per_90", 0)
    xa = row.get("expected_assists_per_90", 0)
    xgc = row.get("expected_goals_conceded_per_90", 0)
    saves = row.get("saves_per_90", 0)
    defcon = row.get("defensive_contribution_per_90", 0)

    base = 0

    # --- GOALS + ASSISTS (position-scaled) ---
    # if pos == "Goalkeeper":
    #     base = xg * 10 + xa * 3
    if pos == "Defender":
        base = xg * 6 + xa * 3
    elif pos == "Midfielder":
        base = xg * 5 + xa * 3
    elif pos == "Forward":
        base = xg * 4 + xa * 3

    # --- CLEAN SHEETS ---
    if pos == "Goalkeeper":
        clean_sheet_pts = row["clean_sheets_per_90"] * 5
    elif pos == "Defender":
        clean_sheet_pts = row["clean_sheets_per_90"] * 4
    elif pos == "Midfielder":
        clean_sheet_pts = row["clean_sheets_per_90"] * 1
    else:
        clean_sheet_pts = 0

    # --- GOALS CONCEDED  ---
    if pos in ["Goalkeeper", "Defender"]:
        conceded_penalty = -0.5 * xgc
    else:
        conceded_penalty = 0

    # --- SAVE POINTS (GK only: 1 per 3 saves) ---
    save_pts = 0
    if pos == "Goalkeeper":
        save_pts = saves / 3

    # --- DEFENSIVE BONUS ---
    if pos == "Defender":
        threshold = 10
    elif pos in ["Midfielder", "Forward"]:
        threshold = 12
    else:
        threshold = None
    if threshold is None:
        defcon_bonus = 0
    else:
        # convert per-90 defcon into expected per-match value
        defcon_match = defcon * row["minutes_factor"]
        
        # probability of hitting match-level threshold
        prob_hit = min(defcon_match / threshold, 1)
    
        # 2-point cap
        defcon_bonus = 2 * prob_hit    

    # ---- APPEARANCE POINTS ----
    mins_est = row["minutes_factor"] * 90

    if mins_est >= 60:
        appearance_pts = 2
    elif mins_est > 0:
        appearance_pts = 1
    else:
        appearance_pts = 0

    return (
        base +
        clean_sheet_pts +
        save_pts +
        defcon_bonus +
        conceded_penalty +
        appearance_pts
    )

POSITION_LIMITS = {"Goalkeeper": 16, "Defender": 40, "Midfielder": 40, "Forward": 24}

def compute_vorp_dynamic(df, metric=""):
    output = []

    for pos, limit in POSITION_LIMITS.items():
        subset = df[df["position"] == pos].copy()
        subset = subset.sort_values(metric, ascending=False)

        if len(subset) < limit:
            baseline = subset[metric].min()
        else:
            baseline = subset.iloc[limit - 1][metric]

        print(pos, len(subset), baseline)

        subset["baseline"] = baseline
        subset["VORP"] = subset[metric] - baseline
        subset["above_replacement"] = subset["VORP"] > 0

        output.append(subset)

    return pd.concat(output, ignore_index=True)


count    667.000000
mean       0.418948
std        0.390409
min        0.000000
25%        0.000000
50%        0.362604
75%        0.819327
max        1.000000
Name: minutes_factor, dtype: float64


In [88]:
MIN_MINUTES = 15   # expand as season progresses
merged = merged[merged["minutes"] >= MIN_MINUTES].copy()

merged["current_x_fpl_p90"] = merged.apply(expected_fpl_p90, axis=1).fillna(0)
merged["bps_GW"] = merged["bps"] / LAST_GW

for col in ["current_x_fpl_p90", "points_per_game", "bps_GW", "last_points_per_game", "team_score", "fixture_score"]:
    merged[col + "_scaled"] = scale_global(merged, col)

print(merged["fixture_score"].describe())

status_multiplier = {"a": 1.0, "d": 0.9, "i": 0.3, "s": 0.3}
merged["availability_factor"] = merged["status"].map(status_multiplier).fillna(1.0)

merged["set_piece_score"] = 0.0
merged.loc[merged["penalties_order"] == 1, "set_piece_score"] += 0.15
merged.loc[merged["direct_freekicks_order"] == 1, "set_piece_score"] += 0.08
merged.loc[merged["corners_and_indirect_freekicks_order"] == 1, "set_piece_score"] += 0.05

#TODO bake the set piece score into secondary position_score function

w_current = 0.8
w_last = 0.2

merged["overall_expected_return"] = (
    w_current * (
        0.43 * merged["current_x_fpl_p90_scaled"] +
        0.37 * merged["points_per_game_scaled"] +
        0.11 * merged["bps_GW_scaled"] +
        0.09 * merged["team_score_scaled"]
    ).fillna(0) +
    w_last * merged["last_points_per_game_scaled"].fillna(0)
)

merged["weekly_expected_return"] = (
    w_current * (
        0.34 * merged["current_x_fpl_p90_scaled"] +
        0.29 * merged["points_per_game_scaled"] +
        0.07 * merged["bps_GW_scaled"] +
        0.22 * merged["fixture_score_scaled"].fillna(0)+
        0.08 * merged["team_score_scaled"]
    ).fillna(0) +
    w_last * merged["last_points_per_game_scaled"].fillna(0)
)

# --- apply stability + availability modifiers ---
merged["weekly_expected_return"] = merged["weekly_expected_return"] * merged["minutes_factor"]
merged["overall_expected_return"] = merged["overall_expected_return"] * merged["minutes_factor"]

merged["weekly_expected_return"] = np.where(
    merged["weekly_expected_return"] > 0,
    merged["weekly_expected_return"] * merged["availability_factor"],   # positive: multiply (penalty works)
    merged["weekly_expected_return"]                                     # negative: leave as-is (don't flip)
)

merged["buy_low"] = merged["current_x_fpl_p90_scaled"] - merged["points_per_game_scaled"]
merged["sell_high"] = merged["points_per_game_scaled"] - merged["current_x_fpl_p90_scaled"]

print(merged["buy_low"].describe())

count    406.000000
mean       0.256206
std        0.327948
min       -0.635646
25%        0.058684
50%        0.270964
75%        0.432410
max        1.000000
Name: fixture_score, dtype: float64
count    406.000000
mean      -0.247407
std        0.550237
min       -3.555320
25%       -0.550045
50%       -0.250677
75%        0.071818
max        2.294207
Name: buy_low, dtype: float64


In [89]:
weekly = compute_vorp_dynamic(merged, metric="weekly_expected_return")[["player_id","VORP"]].rename(columns={"VORP":"weekly_VORP"})
overall = compute_vorp_dynamic(merged, metric="overall_expected_return")[["player_id","VORP"]].rename(columns={"VORP":"overall_VORP"})

board = merged.merge(weekly, on="player_id").merge(overall, on="player_id")
board = board.sort_values("overall_VORP", ascending=False)

Goalkeeper 24 0.009440403018263729
Defender 144 0.2937663971172887
Midfielder 189 0.24915125609400846
Forward 49 -0.09495556119923339
Goalkeeper 24 -0.07949533010726441
Defender 144 0.2615975118314408
Midfielder 189 0.3522556303853357
Forward 49 -0.030364749904660635


In [90]:
BASE_URL = "https://draft.premierleague.com/api"

names = ["Tzolakis","M.Sangaré","Yalcouyé","Araujo","Tzolis"]

all_boards = []

for key, league in config.LEAGUES.items():
    LEAGUE_ID = league["league_id"]
    MY_ENTRY_ID = league["entry_id"]
    league_name = league["name"]          # "Gegen" / "Dalot"

    url = f"{BASE_URL}/league/{LEAGUE_ID}/element-status"
    raw = requests.get(url).json()
    league_players = pd.json_normalize(raw["element_status"])

    league_players["owner"] = pd.to_numeric(league_players["owner"], errors="coerce").astype("Int64")
    league_players["is_available"] = league_players["status"].astype(str) == "a"
    league_players["is_mine"] = league_players["owner"] == MY_ENTRY_ID
    league_players["is_taken"] = (~league_players["is_available"]) & (~league_players["is_mine"])
    league_players = league_players.rename(columns={"element": "player_id"})

    mine_set  = league.get("known_mine", set())
    taken_set = league.get("known_taken", set())
    avail_set = league.get("known_available", set())
    
    # mine / taken first
    league_players.loc[league_players["player_id"].isin(mine_set),  "is_mine"] = True
    league_players.loc[league_players["player_id"].isin(taken_set), "is_taken"] = True
    league_players.loc[league_players["player_id"].isin(mine_set | taken_set), "is_available"] = False

    # known_available LAST — fully reset to available, override everything
    league_players.loc[league_players["player_id"].isin(avail_set), ["is_mine","is_taken"]] = False
    league_players.loc[league_players["player_id"].isin(avail_set), "is_available"] = True

    league_players["is_mine"] = league_players["is_mine"].fillna(False).astype(bool)
    league_players["is_taken"] = league_players["is_taken"].fillna(False).astype(bool)

    check = league_players[league_players["player_id"].isin({557,565,572,579,605,608})]
    print(config.LEAGUES["gegenpressure"].keys())
    print(league_name)
    print(check[["player_id","is_mine","is_available","is_taken"]].to_string(index=False))

    flags = league_players[["player_id", "is_mine", "is_available", "is_taken"]].rename(columns={
        "is_mine":      f"mine_{league_name}",
        "is_available": f"available_{league_name}",
        "is_taken":     f"taken_{league_name}",
    })
    all_boards.append(flags)



dict_keys(['name', 'entry_id', 'league_id', 'league_size', 'format', 'known_taken', 'known_mine', 'known_available'])
Gegen
 player_id  is_mine  is_available  is_taken
       572    False         False      True
       557    False         False      True
       565    False         False      True
       579    False         False      True
       605     True         False     False
       608    False          True     False
dict_keys(['name', 'entry_id', 'league_id', 'league_size', 'format', 'known_taken', 'known_mine', 'known_available'])
Dalot
 player_id  is_mine  is_available  is_taken
       557    False         False      True
       565    False         False      True
       572    False         False      True
       579    False          True     False
       605    False         False      True
       608    False          True     False


In [91]:
enriched = board.copy()
for flags in all_boards:
    enriched = enriched.merge(flags, on="player_id", how="left")

enriched["mins_per_gw"] = enriched["minutes"] / LAST_GW

enriched = enriched.rename(columns={
    "current_x_fpl_p90": "x_fpl_p90",   # what the HTML expects
    "defensive_contribution_per_90": "defcon_p90",
    "expected_goal_involvements_per_90": "xgi_p90"
})

print(enriched.columns.tolist())

enriched.to_csv("weekly_board_enriched.csv", index=False, encoding="utf-8-sig")


['web_name', 'position', 'team_name', 'total_points', 'points_per_game', 'minutes', 'starts', 'bonus', 'bps', 'form', 'expected_goals_per_90', 'expected_assists_per_90', 'xgi_p90', 'expected_goals_conceded_per_90', 'defensive_contribution', 'defcon_p90', 'tackles', 'clearances_blocks_interceptions', 'recoveries', 'penalties_order', 'corners_and_indirect_freekicks_order', 'direct_freekicks_order', 'saves_per_90', 'clean_sheets_per_90', 'goals_conceded_per_90', 'player_id', 'team_code', 'status', 'chance_of_playing_next_round', 'team_attack_score', 'team_defence_score', 'team_score', 'fixture_attack_score', 'fixture_defence_score', 'fixture_score', 'fixture_score_100', 'last_points_per_game', 'last_x_fpl_p90', 'last_total_points', 'last_bonus_p90', 'last_minutes', 'last_defcon_p90', '_merge', 'minutes_factor', 'x_fpl_p90', 'bps_GW', 'current_x_fpl_p90_scaled', 'points_per_game_scaled', 'bps_GW_scaled', 'last_points_per_game_scaled', 'team_score_scaled', 'fixture_score_scaled', 'availabil